# Gene-set control: PGKB 909개가 세포 표현에서 특별한가

성능 자체(nested CV RMSE, 무작위 909개 5회 재추출)는 `results_summary.ipynb`의
통합 표가 근거다. 이 노트북은 그 결과를 **세포 수준의 거리·관계 구조**로만 범위를
좁혀 확인한다. 주장은 하나다:

> 909개 유전자로 세포를 표현할 때, PGKB 큐레이션 셋은 동일 크기 무작위 셋보다
> 특별한 세포 간 구조를 담고 있지 않다.

1. **설계 검증** — 두 셋은 겹치는 유전자가 없고 모델 크기도 같다
2. **유전자 수준으로는 분명히 다른 셋** — 귀무 상황이 아님을 먼저 확인
3. **세포 간 거리 구조** — 세포쌍 380,628개, 모달리티별
4. **주성분 부분공간 중첩** — 세포 점수 상위 20축, 귀무 수준 포함

약물 반응을 평균으로 뭉개는 분석(세포 평균 IC50 예측, 예측값 pooled 상관)은
정보를 과하게 지우므로 여기서 다루지 않는다.


In [1]:
# 이 노트북은 results_summary.ipynb와 독립 실행된다 (공용 셋업만 복제).
import json, glob, os
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)

def find_results_dir(start=None):
    start = Path(start or os.getcwd()).resolve()
    for base in [start, *start.parents]:
        for rel in ['scripts/Results', 'omicsdrp/scripts/Results', 'Results']:
            p = base / rel
            if p.is_dir():
                return str(p)
    raise FileNotFoundError('Results dir not found from ' + str(start))

RESULTS_DIR = find_results_dir()
print('RESULTS_DIR =', RESULTS_DIR)


RESULTS_DIR = /project/OmicsDRP_Review/omicsdrp/scripts/Results


In [2]:
import sys
sys.path.insert(0, str(Path(RESULTS_DIR).parents[1] / 'src'))
import torch
DATA = Path(RESULTS_DIR).parents[2] / 'data'

def gene_dict(gene_set):
    if gene_set == 'pgkb':
        return torch.load(DATA / 'PGKB_Gene_data_dict.pth')
    return torch.load(DATA / 'gene_dicts' / f"random_{gene_set.split(':')[1]}.pth")

SETS = ['pgkb'] + [f'random:{i}' for i in range(1, 6)]
dicts = {g: gene_dict(g) for g in SETS}
pgkb_genes = set(dicts['pgkb'])
print('gene set        n_genes  PGKB와 겹침  다른 draw와 평균 겹침')
for g in SETS:
    gs = set(dicts[g])
    others = [len(gs & set(dicts[h])) for h in SETS if h != g and h != 'pgkb']
    print(f'  {g:12s} {len(gs):7d} {len(gs & pgkb_genes):11d}'
          f'{(np.mean(others) if others else float("nan")):18.1f}')

def n_params(tag):
    for line in open(Path(RESULTS_DIR) / tag / 'events.jsonl'):
        d = json.loads(line)
        if d.get('kind') == 'model_info':
            return d['n_params']

tags = {}
for cfg_path in glob.glob(os.path.join(RESULTS_DIR, '*', 'config.json')):
    cfg = json.load(open(cfg_path))
    key = (cfg.get('gene_set', 'pgkb'), cfg['split_mode'],
           '+'.join(cfg.get('noise_omics') or []),
           '+'.join(sorted(cfg['omics'])), cfg['cell_encoder'], cfg['drug_encoder'])
    tags[key] = os.path.basename(os.path.dirname(cfg_path))

ALL4, ATT, MOR = '+'.join(sorted(['SNP', 'MET', 'CNV', 'RNA'])), 'attention', 'morgan'
def tag_of(gene_set='pgkb', split='mixed', noise=''):
    return tags[(gene_set, split, noise, ALL4, ATT, MOR)]
print('\n파라미터 수 (mixed):')
for g in SETS:
    t = tag_of(g)
    print(f'  {g:12s} {n_params(t):,}   tag={t}')


gene set        n_genes  PGKB와 겹침  다른 draw와 평균 겹침
  pgkb             909         909               0.0
  random:1         909           0              56.8
  random:2         909           0              59.0
  random:3         909           0              60.8
  random:4         909           0              58.2
  random:5         909           0              57.2

파라미터 수 (mixed):
  pgkb         7,758,537   tag=SNP+MET+CNV+RNA__attention__morgan__mixed__c94ea3
  random:1     7,758,537   tag=random1__SNP+MET+CNV+RNA__attention__morgan__mixed__181781
  random:2     7,758,537   tag=random2__SNP+MET+CNV+RNA__attention__morgan__mixed__520428
  random:3     7,758,537   tag=random3__SNP+MET+CNV+RNA__attention__morgan__mixed__e8cda9
  random:4     7,758,537   tag=random4__SNP+MET+CNV+RNA__attention__morgan__mixed__d93f18
  random:5     7,758,537   tag=random5__SNP+MET+CNV+RNA__attention__morgan__mixed__7923b5


In [3]:
# ---- STEP 2. the sets really are different genes (single-gene statistics) ----
# 유전자 수준에서는 두 셋이 확실히 다르다 -- "사실상 같은 유전자를 뽑았다"가 아니다.
COLS = {'SNP': 0, 'MET': 1, 'CNV': 2, 'RNA': 3}

def gene_stats(d):
    M = {k: np.stack([d[g][:, c].numpy() for g in d], 1) for k, c in COLS.items()}
    return {'RNA mean': M['RNA'].mean(), 'RNA sd(cells)': M['RNA'].std(0).mean(),
            'MET mean': M['MET'].mean(), 'CNV sd(cells)': M['CNV'].std(0).mean(),
            'SNP 변이율': (M['SNP'] > 0).mean()}

st = pd.DataFrame({g: gene_stats(dicts[g]) for g in SETS}).T
rand_rows = st.loc[[g for g in SETS if g != 'pgkb']]
st.loc['random 평균'] = rand_rows.mean()
st.loc['PGKB z (random 분포 기준)'] = (st.loc['pgkb'] - rand_rows.mean()) / rand_rows.std()
display(st.round(4))
print('PGKB는 random draw 분포에서 여러 σ 떨어져 있다 -> 유전자 구성 자체는 명확히 다름')


,RNA mean,RNA sd(cells),MET mean,CNV sd(cells),SNP 변이율
pgkb,1.9672,0.8093,0.4720,1.2418,0.1668
random:1,2.1454,0.7167,0.4383,1.1945,0.1551
random:2,2.1016,0.6986,0.4452,1.1994,0.1547
random:3,2.0719,0.6979,0.4492,1.2076,0.1533
random:4,2.0942,0.7058,0.4412,1.2032,0.1485
random:5,2.1061,0.6763,0.4482,1.2502,0.1514
random 평균,2.1039,0.6991,0.4444,1.2110,0.1526
PGKB z (random 분포 기준),-5.1235,7.4433,5.9678,1.3706,5.1785


PGKB는 random draw 분포에서 여러 σ 떨어져 있다 -> 유전자 구성 자체는 명확히 다름


In [4]:
# ---- STEP 3. cell-cell geometry: what the cell encoder actually sees ----
# 모델이 쓰는 건 개별 유전자 값이 아니라 세포 간 상대 배치다.
# 4개 오믹스를 '각각' 확인한다. 모달리티를 이어붙인 표현은 모델이 실제로 융합하는
# 방식(유전자별 임베딩 -> 어텐션)과 달라서 해석이 어려우므로 쓰지 않는다.
# 열마다 표준화 -- 학습 파이프라인의 scale_gene_data와 같은 처리.
from scipy.spatial.distance import pdist
from scipy.stats import spearmanr, pearsonr
from sklearn.preprocessing import StandardScaler

def raw_matrix(d, mod):
    """모달리티 하나를 [n_cell, 909] 원시 행렬로 (표준화 없음)."""
    return np.stack([d[g][:, COLS[mod]].numpy() for g in d], 1)

def omics_matrix(d, mod):
    """raw_matrix + 유전자(열)별 centering & scaling.
    StandardScaler는 열마다 독립이므로 '해당 모달리티의 유전자별' 표준화가 된다
    -- 학습 파이프라인 scale_gene_data가 (유전자, 모달리티) 열 단위로 하는 것과 동일.
    STEP 3/4는 라벨을 안 쓰는 진단이라 전체 873셀로 적합해도 무해하다.
    지도학습인 STEP 5는 fold 내부에서 train 세포만으로 다시 적합한다."""
    return StandardScaler().fit_transform(raw_matrix(d, mod))

MODS = ['SNP', 'MET', 'CNV', 'RNA']

def cell_geometry(d, mod):
    return pdist(omics_matrix(d, mod), metric='correlation')

print('세포쌍 개수:', f'{873*872//2:,}')
print('\nomics    PGKB vs random (평균, 범위)      random끼리      차이')
geo_rows = []
for name in MODS:
    vec = {g: cell_geometry(dicts[g], name) for g in SETS}
    cross = [pearsonr(vec['pgkb'], vec[g])[0] for g in SETS[1:]]
    within = [pearsonr(vec[a], vec[b])[0]
              for i, a in enumerate(SETS[1:]) for b in SETS[1 + i + 1:]]
    geo_rows.append(dict(omics=name, PGKB_vs_random=np.mean(cross),
                         lo=min(cross), hi=max(cross),
                         random_vs_random=np.mean(within),
                         차이=np.mean(within) - np.mean(cross)))
    print(f'  {name:6s}   r = {np.mean(cross):.3f}  [{min(cross):.3f}, {max(cross):.3f}]'
          f'          {np.mean(within):.3f}      {np.mean(within)-np.mean(cross):+.3f}')
display(pd.DataFrame(geo_rows).round(4))
print('PGKB-random 일치도가 random-random 일치도와 비슷하면(차이 작음),')
print('PGKB 셋이 특별한 세포 배치를 담고 있지 않다는 뜻이다.')


세포쌍 개수: 380,628

omics    PGKB vs random (평균, 범위)      random끼리      차이


  SNP      r = 0.233  [0.229, 0.236]          0.263      +0.030


  MET      r = 0.924  [0.920, 0.927]          0.946      +0.022


  CNV      r = 0.904  [0.894, 0.910]          0.939      +0.035


  RNA      r = 0.916  [0.915, 0.917]          0.937      +0.021


,omics,PGKB_vs_random,lo,hi,random_vs_random,차이
0,SNP,0.2332,0.2292,0.2359,0.2632,0.0299
1,MET,0.9236,0.9204,0.9271,0.9460,0.0224
2,CNV,0.9036,0.8939,0.9100,0.9386,0.0350
3,RNA,0.9163,0.9147,0.9171,0.9373,0.0211


PGKB-random 일치도가 random-random 일치도와 비슷하면(차이 작음),
PGKB 셋이 특별한 세포 배치를 담고 있지 않다는 뜻이다.


In [5]:
# ---- STEP 4. PC subspace overlap: are the leading axes the same axes? ----
# 모달리티 4개에서 각각 상위 K개 PC 부분공간을 비교.
#
# 무엇의 주성분인가: [873 세포 x 909 유전자] 표준화 행렬의 주성분.
#   - loading(components_) = [K x 909], 유전자 공간의 방향. 두 유전자셋은 겹치는
#     유전자가 0개라 서로 다른 공간에 있어 '직접 비교 불가'.
#   - score(transform)     = [873 x K], 세포마다 축 위의 좌표. 세포는 같은 873개이므로
#     '비교 가능'. 그래서 비교 대상은 세포 점수 쪽이다.

from sklearn.decomposition import PCA
from sklearn.cross_decomposition import CCA

K = 20
rng = np.random.default_rng(0)
summary_rows = []
cc_by_mod = {}
for name in MODS:
    Z = {g: omics_matrix(dicts[g], name) for g in SETS}
    ev = {g: PCA(K).fit(Z[g]).explained_variance_ratio_ for g in SETS}
    P = {g: PCA(K).fit_transform(Z[g]) for g in SETS}

    def cc_vs(Pb):
        cc = CCA(n_components=K, max_iter=3000).fit(P['pgkb'], Pb)
        Xc, Yc = cc.transform(P['pgkb'], Pb)
        return np.array([abs(np.corrcoef(Xc[:, i], Yc[:, i])[0, 1]) for i in range(K)])

    obs = np.vstack([cc_vs(P[g]) for g in SETS[1:]])
    cc_by_mod[name] = obs.mean(0)
    perm = Z['random:1'][rng.permutation(Z['random:1'].shape[0])]
    null = np.vstack([cc_vs(PCA(K).fit_transform(StandardScaler().fit_transform(M)))
                      for M in (perm, rng.standard_normal(Z['pgkb'].shape))])
    summary_rows.append(dict(
        omics=name,
        PGKB_PC1=ev['pgkb'][0], PGKB_PC1_20=ev['pgkb'].sum(),
        random_PC1=np.mean([ev[g][0] for g in SETS[1:]]),
        random_PC1_20=np.mean([ev[g].sum() for g in SETS[1:]]),
        CC1=obs[:, 0].mean(), CC10=obs[:, 9].mean(),
        축_0p9이상=np.mean((obs >= 0.9).sum(1)),
        귀무_CC1=null[:, 0].mean(), 귀무_평균=null.mean(),
        귀무_축_0p9이상=np.mean((null >= 0.9).sum(1))))
display(pd.DataFrame(summary_rows).round(3))
print('축별 정준상관 (PGKB vs random 5draw 평균):')
display(pd.DataFrame(cc_by_mod, index=[f'CC{i+1}' for i in range(K)]).round(3))
print('귀무 수준(CC1 ~0.30, 0.9이상 0개)을 크게 넘으면 실제 공유 구조다.')

# --- 참고: CCA(부분공간 정렬) vs PC 직접 대응(회전 허용 안 함) 의 차이 ---
Zr = {g: omics_matrix(dicts[g], 'RNA') for g in ['pgkb', 'random:1']}
Sa = PCA(10).fit_transform(Zr['pgkb'])          # 세포 점수 [873, 10]
Sb = PCA(10).fit_transform(Zr['random:1'])
Cm = np.abs(np.corrcoef(Sa.T, Sb.T)[:10, 10:])
print('\n[RNA, PGKB vs random:1] PC 직접 대응 |corr| (회전 없이 PC_i vs PC_i):')
print('  ' + ' '.join(f'PC{i+1}={Cm[i, i]:.2f}' for i in range(10)))
print('  각 PGKB PC의 최적 상대: '
      + ' '.join(f'PC{i+1}->PC{Cm[i].argmax()+1}({Cm[i].max():.2f})' for i in range(10)))
print('  -> 앞 5축은 번호까지 대응되지만 이후는 순서가 섞인다.')
print('     CCA가 0.9 이상을 12축까지 주는 것은 "축 순서까지 같다"가 아니라 총 12차원의 축이 공유되는 현상을 보인다는 것임')


,omics,PGKB_PC1,PGKB_PC1_20,random_PC1,random_PC1_20,CC1,CC10,축_0p9이상,귀무_CC1,귀무_평균,귀무_축_0p9이상
0,SNP,0.163,0.351,0.156,0.343,0.996,0.850,7.0,0.440,0.118,0.0
1,MET,0.282,0.626,0.262,0.600,0.998,0.938,13.8,0.268,0.128,0.0
2,CNV,0.462,0.692,0.472,0.700,0.999,0.955,13.8,0.270,0.127,0.0
3,RNA,0.084,0.455,0.105,0.459,0.993,0.938,12.0,0.276,0.131,0.0


축별 정준상관 (PGKB vs random 5draw 평균):


,SNP,MET,CNV,RNA
CC1,0.996,0.998,0.999,0.993
CC2,0.975,0.993,0.988,0.989
CC3,0.958,0.991,0.985,0.987
CC4,0.948,0.988,0.982,0.983
CC5,0.930,0.986,0.978,0.976
CC6,0.918,0.978,0.975,0.967
CC7,0.903,0.970,0.971,0.958
CC8,0.893,0.961,0.964,0.950
CC9,0.871,0.953,0.961,0.944
CC10,0.850,0.938,0.955,0.938


귀무 수준(CC1 ~0.30, 0.9이상 0개)을 크게 넘으면 실제 공유 구조다.

[RNA, PGKB vs random:1] PC 직접 대응 |corr| (회전 없이 PC_i vs PC_i):
  PC1=0.85 PC2=0.86 PC3=0.91 PC4=0.86 PC5=0.81 PC6=0.31 PC7=0.36 PC8=0.61 PC9=0.56 PC10=0.37
  각 PGKB PC의 최적 상대: PC1->PC1(0.85) PC2->PC2(0.86) PC3->PC3(0.91) PC4->PC4(0.86) PC5->PC5(0.81) PC6->PC9(0.38) PC7->PC6(0.62) PC8->PC8(0.61) PC9->PC9(0.56) PC10->PC10(0.37)
  -> 앞 5축은 번호까지 대응되지만 이후는 순서가 섞인다.
     CCA가 0.9 이상을 12축까지 주는 것은 "축 순서까지 같다"가 아니라 총 12차원의 축이 공유되는 현상을 보인다는 것임
